# MD starter — harmonic FCC crystal, velocity-Verlet

Workflow:

1. Place argon-mass atoms on a 4 × 4 × 4 unit-cell FCC lattice (256 atoms).
2. Pre-compute the harmonic coupling matrix `K` from neighbour pairs.
3. Trace one velocity-Verlet step as two matmuls + Verlet algebra.
4. Preflight, submit, fetch updated positions/velocities.
5. Sanity-check against the NumPy reference (energy conservation).

The harmonic model is the **starter floor**. Replacing it with a true
Lennard-Jones force is the work — see `lj_forces.py` for the tracing
scaffold and `baseline.py:lj_step_numpy` for the reference.

Required env vars: `UNIQX_API_KEY`, `UNIQX_GATEWAY`.

## 1 · Setup

In [ ]:
import os

import numpy as np
import uniqx
from harmonic_step import build_coupling_matrix, build_harmonic_step_module, fcc_lattice

GATEWAY = os.environ.get("UNIQX_GATEWAY", "localhost:50050")
API_KEY = os.environ.get("UNIQX_API_KEY")  # required for api.oriqx.com; None for local
client = uniqx.connect(GATEWAY, api_key=API_KEY)
print("uniqx", uniqx.__version__, "— gateway:", GATEWAY)

## 2 · Workload — FCC argon crystal

32 atoms on a 2 × 2 × 2 FCC lattice. Mass and length units chosen so
the harmonic frequency is order-1 in the timestep.

In [ ]:
positions = fcc_lattice(n_per_side=2, a=3.6)
n_atoms = len(positions)
print(f"atoms      : {n_atoms}")

rng = np.random.default_rng(42)
velocities = (rng.standard_normal((n_atoms, 3)) * 0.01).tolist()

MASS = 40.0          # argon-ish, in atomic mass units
DT = 0.005
CUTOFF = 5.0
SPRING_K = 1.0

K = build_coupling_matrix(positions, cutoff=CUTOFF, spring_k=SPRING_K)
module, runtime_inputs, meta = build_harmonic_step_module(
    positions, velocities, MASS, K, DT
)
print(f"traced ops : {len(module.functions[0].ops)}")
print(f"dim        : {meta['dim']} (= 3 * n_atoms)")

## 3 · Preflight

In [ ]:
options = uniqx.preflight(module, client=client)
print(options.summary())

In [ ]:
choice = options.recommended
print(f"Selected: {choice['label']}")
print(f"  node assignment : {choice.get('node_assignments', {})}")

## 4 · Submit & retrieve

In [ ]:
job_id = uniqx.submit(
    module,
    client=client,
    preflight_job_id=options.job_id,
    option_idx=choice["_idx"],
    runtime_inputs=runtime_inputs,
)
result = uniqx.get(job_id, client=client, timeout=300)

payload = result.get("payload") or result.get("result_payload") or b""
if isinstance(payload, str):
    payload = payload.encode("utf-8", errors="replace")

outputs = []
for line in payload.decode("utf-8", errors="replace").splitlines():
    parsed = uniqx.parse_buffer_view(line.strip())
    if parsed is not None:
        outputs.append(parsed)

x_new = np.array(outputs[0][2])
v_new = np.array(outputs[1][2])
kinetic_uniqx = outputs[2][2][0]
print(f"received {len(x_new)} position components, {len(v_new)} velocity components")
print(f"kinetic energy (uniqx): {kinetic_uniqx:.6e}")

## 5 · Baseline — NumPy reference

In [ ]:
from baseline import harmonic_step_numpy, total_energy_harmonic

K_np = np.array(K)
x0 = np.array([v for atom in positions for v in atom])
v0 = np.array([v for atom in velocities for v in atom])
x_ref, v_ref, kinetic_ref = harmonic_step_numpy(x0, v0, K_np, MASS, DT)

x_diff = float(np.linalg.norm(x_new - x_ref) / max(np.linalg.norm(x_ref), 1e-12))
v_diff = float(np.linalg.norm(v_new - v_ref) / max(np.linalg.norm(v_ref), 1e-12))
print(f"position relative L2 diff : {x_diff:.2e}")
print(f"velocity relative L2 diff : {v_diff:.2e}")
print(f"kinetic (numpy ref)       : {kinetic_ref:.6e}")

## 6 · Discussion (your job)

1. **Scale to 4 × 4 × 4 (256 atoms).** Re-run preflight. Where does the
   matmul start to favour GPU?
2. **Replace harmonic with Lennard-Jones.** `lj_forces.py` has the
   scaffold; `baseline.py:lj_step_numpy` is the reference. Compare
   total-energy drift over 100 steps against the harmonic baseline.
3. **Compose 100 steps into one traced module.** Hint: `ops.control_flow`
   exposes `fori_loop`. The gateway gets a much richer graph to optimise.